# Deepfake Detection — 1D CNN + Optuna Hyperparameter Optimization
ASVspoof 2019 LA dataset

In [2]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q pytorch-optimizer optuna onnx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.4/287.4 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 98.0 MB/s eta 0:00:00


In [3]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import os
import random
import shutil
from contextlib import redirect_stdout

import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, precision_score, recall_score, f1_score
)

from pytorch_optimizer import Ranger

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

import onnx

from google.colab import drive

In [4]:
# ── Cell 3: Device & seed ──────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print(torch.cuda.get_device_name(0))

def set_seed(seed=22):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(22)

Using device: cuda
NVIDIA A100-SXM4-80GB


In [5]:
# ── Cell 4: Mount Drive & load data ───────────────────────────────────────────
drive.mount('/content/drive')

DEV_PATH   = ('/content/drive/MyDrive/ASV_Final_Features_dev.pkl',  'dev')
EVAL_PATH  = ('/content/drive/MyDrive/ASV_Final_Features_eval.pkl', 'eval')
TRAIN_PATH = ('/content/drive/MyDrive/ASV_Final_Features_train.pkl','train')

def load_dataset(path):
    df = pd.read_pickle(path[0])
    print(f'DataFrame loaded for {path[1]} set. '
          f'Total samples: {len(df)}, shape: {df.shape}')
    return df

train_df = load_dataset(TRAIN_PATH)
dev_df   = load_dataset(DEV_PATH)
eval_df  = load_dataset(EVAL_PATH)

print('\nLabel distribution:')
print('Train:', train_df['label'].value_counts().to_dict())
print('Dev:  ', dev_df['label'].value_counts().to_dict())
print('Eval: ', eval_df['label'].value_counts().to_dict())

Mounted at /content/drive
DataFrame loaded for train set. Total samples: 25380, shape: (25380, 5)
DataFrame loaded for dev set. Total samples: 24844, shape: (24844, 5)
DataFrame loaded for eval set. Total samples: 71237, shape: (71237, 5)

Label distribution:
Train: {1: 22800, 0: 2580}
Dev:   {1: 22296, 0: 2548}
Eval:  {1: 63882, 0: 7355}


In [6]:
# ── Cell 5: Prepare tensors & normalization stats ──────────────────────────────
def prepare_tensors(df, feature_col='1dcnn_features', label_col='label'):
    X = np.stack(df[feature_col].values).transpose(0, 2, 1).astype(np.float32)
    y = df[label_col].values
    return torch.from_numpy(X), torch.from_numpy(y)

X_train_tr, y_train_tr = prepare_tensors(train_df)
X_dev_tr,   y_dev_tr   = prepare_tensors(dev_df)
X_eval_tr,  y_eval_tr  = prepare_tensors(eval_df)

# Normalization stats (computed on train only)
epsilon = 1e-4
x_means = X_train_tr.mean(dim=(0, 2), keepdim=True)
x_stds  = X_train_tr.std(dim=(0, 2),  keepdim=True) + epsilon

print(f'Train : {X_train_tr.shape}, {y_train_tr.shape}')
print(f'Dev   : {X_dev_tr.shape},   {y_dev_tr.shape}')
print(f'Eval  : {X_eval_tr.shape},  {y_eval_tr.shape}')

Train : torch.Size([25380, 40, 126]), torch.Size([25380])
Dev   : torch.Size([24844, 40, 126]),   torch.Size([24844])
Eval  : torch.Size([71237, 40, 126]),  torch.Size([71237])


In [7]:
# ── Cell 6: Model components ───────────────────────────────────────────────────
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.SiLU(),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)


class AttentivePooling(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.attention = nn.Linear(in_channels, 1)

    def forward(self, x):
        x_trans  = x.transpose(1, 2)              # (B, T, C)
        weights  = torch.softmax(self.attention(x_trans), dim=1)  # (B, T, 1)
        pooled   = torch.sum(x_trans * weights, dim=1)            # (B, C)
        return pooled

In [8]:
# ── Cell 7: 1D CNN model (Optuna-parameterised) ────────────────────────────────
class Deepfake_1DCNN(nn.Module):
    """
    Parameters
    ----------
    means, stds   : normalization buffers (from train set)
    out_channels  : number of Conv1d output channels
    kernel_size   : Conv1d kernel size
    dilation      : Conv1d dilation factor
    dropout       : dropout rate in classification head
    fc1_out       : hidden size of the first FC layer

    The pooling strategy is fixed (mean + std + max + attentive),
    so the FC input is always out_channels * 4.
    """
    def __init__(self, means, stds,
                 out_channels=32,
                 kernel_size=5,
                 dilation=4,
                 dropout=0.4,
                 fc1_out=16):
        super().__init__()

        self.register_buffer('means', means.detach().clone().view(1, 40, 1))
        self.register_buffer('stds',  stds.detach().clone().view(1, 40, 1))

        # ── Conv block ─────────────────────────────────────────────────────────
        # padding = dilation * (kernel_size - 1) // 2 keeps output length same
        padding = dilation * (kernel_size - 1) // 2
        self.conv1    = nn.Conv1d(in_channels=40, out_channels=out_channels,
                                  kernel_size=kernel_size, stride=1,
                                  padding=padding, dilation=dilation)
        self.bn1      = nn.BatchNorm1d(out_channels)
        self.silu     = nn.SiLU()
        self.se       = SEBlock(out_channels)
        self.att_pool = AttentivePooling(out_channels)

        # ── Classification head ────────────────────────────────────────────────
        # mean + std + max + attentive = out_channels * 4
        self.fc1      = nn.Linear(out_channels * 4, fc1_out)
        self.dropout  = nn.Dropout(dropout)
        self.fc2      = nn.Linear(fc1_out, 2)

    def forward(self, x):
        # Normalize
        x = (x - self.means) / (self.stds + 1e-7)

        # Feature extraction
        x = self.silu(self.bn1(self.conv1(x)))   # (B, C, T)

        # SE attention
        x = self.se(x)

        # Pooling — 4 strategies concatenated
        mean_p       = torch.mean(x, dim=2)
        std_p        = torch.std(x,  dim=2)
        max_p, _     = torch.max(x,  dim=2)
        att_p        = self.att_pool(x)
        x = torch.cat((mean_p, std_p, max_p, att_p), dim=1)  # (B, C*4)

        # Classification
        x = self.dropout(self.silu(self.fc1(x)))
        return self.fc2(x)

In [13]:
# ── Cell 8: Evaluation function ────────────────────────────────────────────────
def eval_model(model, dl, device, dataset_name='Dataset', show_plots=True):
    """
    Returns (eer, fig).
    Pass show_plots=False during Optuna trials to suppress all output.
    """
    model.eval()
    all_labels, all_preds, all_scores = [], [], []

    with torch.no_grad():
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            probs = torch.softmax(model(xb), dim=1)
            preds = torch.argmax(probs, dim=1)
            all_labels.extend(yb.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_scores.extend(probs[:, 0].cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_scores = np.array(all_scores)

    acc = accuracy_score(all_labels, all_preds)
    p   = precision_score(all_labels, all_preds, zero_division=0)
    r   = recall_score(all_labels, all_preds, zero_division=0)
    f1  = f1_score(all_labels, all_preds, zero_division=0)

    fpr, tpr, _ = roc_curve(all_labels, all_scores, pos_label=0)
    fnr = 1 - tpr
    eer = fpr[np.nanargmin(np.absolute(fnr - fpr))] * 100

    if show_plots:
        print(f"\n{'='*10} {dataset_name} Results {'='*10}")
        print(f"EER:       {eer:.4f}%")
        print(f"Accuracy:  {acc:.4f}")
        print(f"F1 Score:  {f1:.4f}")
        print(f"Precision: {p:.4f}")
        print(f"Recall:    {r:.4f}")
        print('\nClassification Report:')
        print(classification_report(all_labels, all_preds,
                                    target_names=['Bonafide', 'Spoof']))

        cm = confusion_matrix(all_labels, all_preds)
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                    xticklabels=['Bonafide', 'Spoof'],
                    yticklabels=['Bonafide', 'Spoof'])
        ax.set_title(f'Confusion Matrix: {dataset_name}')
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
        plt.tight_layout()
        plt.show()
        return eer, fig

    return eer, None

In [ ]:
# ── Cell 9: Training loop ──────────────────────────────────────────────────────
def training_loop(epochs, model, loss_fn, opt, train_dl, dev_dl,
                  device, scheduler=None, patience=10, trial=None):
    """
    Returns best dev EER achieved.
    Pass `trial` to enable Optuna pruning.
    """
    best_dev_eer = float('inf')
    epochs_without_improvement = 0
    best_model_state = None

    print(f"{'Epoch':<6} | {'Train Loss':<12} | {'Dev EER':<10} | {'LR':<12}")
    print('-' * 52)

    for epoch in range(epochs):
        # ── Training pass ──────────────────────────────────────────────────────
        model.train()
        running_loss = 0.0

        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            y_pred = model(xb)
            loss   = loss_fn(y_pred, yb)

            opt.zero_grad()
            loss.backward()
            opt.step()

            if scheduler:
                scheduler.step()

            running_loss += loss.item()

        # ── Evaluation (silent) ────────────────────────────────────────────────
        current_dev_eer, _ = eval_model(model, dev_dl, device, show_plots=False)
        avg_loss   = running_loss / len(train_dl)
        current_lr = opt.param_groups[0]['lr']

        print(f"[{epoch:02d}]   | {avg_loss:.4f}       | "
              f"{current_dev_eer:.2f}%     | {current_lr:.6f}")

        # ── Optuna: report + prune check ───────────────────────────────────────
        if trial is not None:
            trial.report(current_dev_eer, epoch)
            if trial.should_prune():
                print(f'[!] Trial pruned at epoch {epoch}')
                raise optuna.exceptions.TrialPruned()

        # ── Early stopping ─────────────────────────────────────────────────────
        if current_dev_eer < best_dev_eer:
            best_dev_eer = current_dev_eer
            epochs_without_improvement = 0
            best_model_state = {k: v.cpu().clone()
                                for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'\n[!] Early stopping triggered at epoch {epoch}.')
                print(f'Best Dev EER: {best_dev_eer:.2f}%')
                break

    if best_model_state:
        model.load_state_dict(best_model_state)
    return best_dev_eer

In [ ]:
# ── Cell 10: Optuna objective ──────────────────────────────────────────────────
def make_dataloaders(batch_size):
    train_ds = TensorDataset(X_train_tr, y_train_tr)
    dev_ds   = TensorDataset(X_dev_tr,   y_dev_tr)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                          pin_memory=True, num_workers=2)
    dev_dl   = DataLoader(dev_ds,   batch_size=batch_size, shuffle=False,
                          pin_memory=True, num_workers=0)
    return train_dl, dev_dl


def objective(trial):
    # ── Search space ───────────────────────────────────────────────────────────
    batch_size    = trial.suggest_categorical('batch_size',    [128, 256, 512])
    lr            = trial.suggest_float('lr',                  1e-4, 1e-2, log=True)
    weight_decay  = trial.suggest_float('weight_decay',        1e-6, 1e-3, log=True)
    out_channels  = trial.suggest_categorical('out_channels',  [16, 32, 64])
    kernel_size   = trial.suggest_categorical('kernel_size',   [3, 5, 7])
    dilation      = trial.suggest_categorical('dilation',      [1, 2, 4])
    dropout       = trial.suggest_float('dropout',             0.1, 0.5)
    fc1_out       = trial.suggest_categorical('fc1_out',       [16, 32, 64])
    optimizer_name = trial.suggest_categorical('optimizer',    ['Ranger', 'Adam'])
    max_lr_factor  = trial.suggest_float('max_lr_factor',      2.0, 10.0)

    set_seed(22)
    train_dl, dev_dl = make_dataloaders(batch_size)

    model = Deepfake_1DCNN(
        x_means, x_stds,
        out_channels=out_channels,
        kernel_size=kernel_size,
        dilation=dilation,
        dropout=dropout,
        fc1_out=fc1_out,
    ).to(device)

    weights = torch.tensor([2.0, 1.0], dtype=torch.float32).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

    if optimizer_name == 'Ranger':
        opt = Ranger(model.parameters(), lr=lr, alpha=0.5, k=6,
                     weight_decay=weight_decay)
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr,
                               weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        opt,
        max_lr=lr * max_lr_factor,
        steps_per_epoch=len(train_dl),
        epochs=50,
        pct_start=0.2,
        div_factor=10,
        final_div_factor=100
    )

    best_eer = training_loop(
        epochs=50,
        model=model,
        loss_fn=loss_fn,
        opt=opt,
        train_dl=train_dl,
        dev_dl=dev_dl,
        device=device,
        scheduler=scheduler,
        patience=10,
        trial=trial,
    )
    return best_eer

In [ ]:
# ── Cell 11: Run Optuna study ──────────────────────────────────────────────────
# Different db file from the CRNN study — keeps records separate.
STUDY_DB   = 'sqlite:////content/drive/MyDrive/optuna_1dcnn.db'
STUDY_NAME = '1dcnn_asvspoof'

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STUDY_DB,
    load_if_exists=True,
    direction='minimize',
    sampler=TPESampler(seed=22, n_startup_trials=10),
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=10),
)

study.optimize(
    objective,
    n_trials=51,
    timeout=3 * 3600,
    gc_after_trial=True,
)

print('\n── Optuna complete ──────────────────────────')
print(f'Best EER   : {study.best_value:.4f}%')
print(f'Best params: {study.best_params}')

[I 2026-04-07 00:51:56,115] Using an existing study with name '1dcnn_asvspoof' instead of creating a new one.


Epoch  | Train Loss   | Dev EER    | LR          
----------------------------------------------------
[00]   | 0.6570       | 46.35%     | 0.000083
[01]   | 0.6155       | 43.50%     | 0.000127
[02]   | 0.5771       | 32.43%     | 0.000195
[03]   | 0.5405       | 16.97%     | 0.000280
[04]   | 0.4639       | 11.92%     | 0.000375
[05]   | 0.3823       | 8.14%     | 0.000470
[06]   | 0.3423       | 6.74%     | 0.000555
[07]   | 0.3233       | 5.52%     | 0.000623
[08]   | 0.3153       | 4.49%     | 0.000666
[09]   | 0.3111       | 4.79%     | 0.000681


[I 2026-04-07 00:52:35,554] Trial 72 pruned. 


[10]   | 0.3083       | 3.63%     | 0.000680
[!] Trial pruned at epoch 10
Epoch  | Train Loss   | Dev EER    | LR          
----------------------------------------------------
[00]   | 0.6417       | 44.87%     | 0.000210
[01]   | 0.5825       | 31.87%     | 0.000319
[02]   | 0.5301       | 15.24%     | 0.000490
[03]   | 0.4225       | 8.98%     | 0.000706
[04]   | 0.3483       | 5.79%     | 0.000945
[05]   | 0.3217       | 4.85%     | 0.001184
[06]   | 0.3131       | 4.18%     | 0.001399
[07]   | 0.3091       | 3.02%     | 0.001570
[08]   | 0.3065       | 3.87%     | 0.001679
[09]   | 0.3053       | 3.91%     | 0.001717
[10]   | 0.3025       | 3.01%     | 0.001714
[11]   | 0.3020       | 2.88%     | 0.001706
[12]   | 0.3005       | 3.08%     | 0.001693
[13]   | 0.2995       | 2.73%     | 0.001675
[14]   | 0.2991       | 2.33%     | 0.001651
[15]   | 0.2985       | 3.10%     | 0.001623
[16]   | 0.2982       | 3.11%     | 0.001590
[17]   | 0.2979       | 2.10%     | 0.001553
[18]   | 0

[I 2026-04-07 00:54:01,428] Trial 73 finished with value: 2.1035163257983496 and parameters: {'batch_size': 128, 'lr': 0.0004727027439809731, 'weight_decay': 4.5262477053124564e-05, 'out_channels': 64, 'kernel_size': 7, 'dilation': 2, 'dropout': 0.3560690676545699, 'fc1_out': 32, 'optimizer': 'Ranger', 'max_lr_factor': 3.631628176940594}. Best is trial 7 with value: 1.426264800861141.


...

[27]   | 0.2934       | 2.28%     | 0.003969

[!] Early stopping triggered at epoch 27.
Best Dev EER: 1.74%
Epoch  | Train Loss   | Dev EER    | LR          
----------------------------------------------------
[00]   | 0.5829       | 17.41%     | 0.001542
[01]   | 0.4149       | 7.86%     | 0.002351
[02]   | 0.3283       | 5.03%     | 0.003610
[03]   | 0.3129       | 2.98%     | 0.005196
[04]   | 0.3105       | 3.75%     | 0.006954
[05]   | 0.3071       | 3.63%     | 0.008712
[06]   | 0.3047       | 3.28%     | 0.010297
[07]   | 0.3028       | 3.38%     | 0.011554
[08]   | 0.3018       | 2.61%     | 0.012360
[09]   | 0.2997       | 1.81%     | 0.012635
[10]   | 0.2971       | 2.08%     | 0.012616
[11]   | 0.2964       | 2.56%     | 0.012557
[12]   | 0.2960       | 2.95%     | 0.012461
[13]   | 0.2951       | 3.41%     | 0.012326
[14]   | 0.2963       | 3.36%     | 0.012154
[15]   | 0.2952       | 2.71%     | 0.011946
[16]   | 0.2959       | 4.73%     | 0.011704
[17]   | 0.2951       |

[I 2026-04-07 01:33:53,740] Trial 100 finished with value: 1.8074991029781127 and parameters: {'batch_size': 128, 'lr': 0.0046870896586815435, 'weight_decay': 7.103766905521592e-05, 'out_channels': 64, 'kernel_size': 7, 'dilation': 1, 'dropout': 0.34757096116006825, 'fc1_out': 32, 'optimizer': 'Ranger', 'max_lr_factor': 2.6958044690489036}. Best is trial 84 with value: 1.125762468604234.


[19]   | 0.2944       | 3.36%     | 0.010785

[!] Early stopping triggered at epoch 19.
Best Dev EER: 1.81%
Epoch  | Train Loss   | Dev EER    | LR          
----------------------------------------------------
[00]   | 0.5712       | 15.11%     | 0.002002
[01]   | 0.3878       | 6.93%     | 0.003052
[02]   | 0.3221       | 4.61%     | 0.004686
[03]   | 0.3113       | 2.47%     | 0.006746
[04]   | 0.3095       | 3.59%     | 0.009028
[05]   | 0.3062       | 3.58%     | 0.011310
[06]   | 0.3043       | 3.16%     | 0.013368


Exception in thread Thread-684 (_pin_memory_loop):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
    do_one_step()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
    r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 122, in get
    return _ForkingPickler.loads(res)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/multiprocessing/reductions.py", line 540, in rebuild_storage_fd
    fd = df.detach()
         ^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/resource

KeyboardInterrupt: 

In [ ]:
# ── Cell 12: Final training with best hyperparameters ─────────────────────────
best = study.best_params
print('Training final model with:', best)

EPOCHS = 200

set_seed(22)
train_dl, dev_dl = make_dataloaders(best['batch_size'])
eval_ds  = TensorDataset(X_eval_tr, y_eval_tr)
eval_dl  = DataLoader(eval_ds, batch_size=best['batch_size'],
                      shuffle=False, pin_memory=True, num_workers=0)

model = Deepfake_1DCNN(
    x_means, x_stds,
    out_channels=best['out_channels'],
    kernel_size=best['kernel_size'],
    dilation=best['dilation'],
    dropout=best['dropout'],
    fc1_out=best['fc1_out'],
).to(device)

weights = torch.tensor([2.0, 1.0], dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.1)

if best['optimizer'] == 'Ranger':
    opt = Ranger(model.parameters(), lr=best['lr'], alpha=0.5, k=6,
                 weight_decay=best['weight_decay'])
else:
    opt = torch.optim.Adam(model.parameters(), lr=best['lr'],
                           weight_decay=best['weight_decay'])

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    opt,
    max_lr=best['lr'] * best['max_lr_factor'],
    steps_per_epoch=len(train_dl),
    epochs=EPOCHS,
    pct_start=0.2,
    div_factor=10,
    final_div_factor=100
)

print(model)

training_loop(
    epochs=EPOCHS,
    model=model,
    loss_fn=loss_fn,
    opt=opt,
    train_dl=train_dl,
    dev_dl=dev_dl,
    device=device,
    scheduler=scheduler,
    patience=15,
    trial=None,
)

Training final model with: {'batch_size': 128, 'lr': 0.00733005223070758, 'weight_decay': 1.3507337007151238e-05, 'out_channels': 64, 'kernel_size': 7, 'dilation': 2, 'dropout': 0.26981219618819724, 'fc1_out': 32, 'optimizer': 'Ranger', 'max_lr_factor': 2.7134491709710105}
Deepfake_1DCNN(
  (conv1): Conv1d(40, 64, kernel_size=(7,), stride=(1,), padding=(6,), dilation=(2,))
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (silu): SiLU()
  (se): SEBlock(
    (avg_pool): AdaptiveAvgPool1d(output_size=1)
    (fc): Sequential(
      (0): Linear(in_features=64, out_features=8, bias=False)
      (1): SiLU()
      (2): Linear(in_features=8, out_features=64, bias=False)
      (3): Sigmoid()
    )
  )
  (att_pool): AttentivePooling(
    (attention): Linear(in_features=64, out_features=1, bias=True)
  )
  (fc1): Linear(in_features=256, out_features=32, bias=True)
  (dropout): Dropout(p=0.26981219618819724, inplace=False)
  (fc2): Linear(in_features=32, ou

np.float64(1.529422317904557)

In [ ]:
# ── Cell 13: Evaluate on Dev set (with confusion matrix) ──────────────────────
dev_eer, dev_fig = eval_model(model, dev_dl, device, 'Development Set', show_plots=True)


========== Development Set Results ==========
EER:       1.5294%
Accuracy:  0.9882
F1 Score:  0.9935
Precision: 0.9871
Recall:    0.9999

Classification Report:
              precision    recall  f1-score   support

    Bonafide       1.00      0.89      0.94      2548
       Spoof       0.99      1.00      0.99     22296

    accuracy                           0.99     24844
   macro avg       0.99      0.94      0.97     24844
weighted avg       0.99      0.99      0.99     24844



In [ ]:
# ── Cell 14: Evaluate on Eval set (with confusion matrix) ─────────────────────
eval_eer, eval_fig = eval_model(model, eval_dl, device, 'Evaluation Set', show_plots=True)

print(f'\nGeneralization gap: {eval_eer - dev_eer:.2f}%')


========== Evaluation Set Results ==========
EER:       8.6472%
Accuracy:  0.8833
F1 Score:  0.9309
Precision: 0.9925
Recall:    0.8765

Classification Report:
              precision    recall  f1-score   support

    Bonafide       0.47      0.94      0.63      7355
       Spoof       0.99      0.88      0.93     63882

    accuracy                           0.88     71237
   macro avg       0.73      0.91      0.78     71237
weighted avg       0.94      0.88      0.90     71237


Generalization gap: 7.12%


In [ ]:
# ── Cell 15: Save model ────────────────────────────────────────────────────────
torch.save(model.state_dict(), '/content/drive/MyDrive/1dcnn_best.pth')
torch.save({'means': x_means, 'stds': x_stds},
           '/content/drive/MyDrive/1dcnn_norm_stats.pth')
print('Model and normalization stats saved to Drive.')

Model and normalization stats saved to Drive.


In [ ]:
# ── Cell 16: Reload model ──────────────────────────────────────────────────────
norm_stats = torch.load('/content/drive/MyDrive/1dcnn_norm_stats.pth')
model = Deepfake_1DCNN(
    norm_stats['means'], norm_stats['stds'],
    out_channels=best['out_channels'],
    kernel_size=best['kernel_size'],
    dilation=best['dilation'],
    dropout=best['dropout'],
    fc1_out=best['fc1_out'],
).to(device)
model.load_state_dict(torch.load('/content/drive/MyDrive/1dcnn_best.pth'))
model.eval()
print('Model reloaded.')

Model reloaded.


---
## Optional: ONNX export

In [ ]:
!pip install -q onnx onnxscript onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 17.3 MB/s eta 0:00:00


In [ ]:
# ── Cell 17: Export to ONNX ────────────────────────────────────────────────────
local_path = 'deepfake_1dcnn_web.onnx'
drive_path = '/content/drive/MyDrive/deepfake_1dcnn_web.onnx'

model.eval()
model.to('cpu')

# Dummy input: (batch=1, 40 MFCCs, 126 frames)
dummy_input = torch.randn(1, 40, 126)

torch.onnx.export(
    model,
    dummy_input,
    local_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input':  {0: 'batch_size', 2: 'audio_length'},
        'output': {0: 'batch_size'}
    }
)

shutil.copy(local_path, drive_path)
print(f'ONNX model exported to Drive: {drive_path}')

# Verify
onnx_model = onnx.load(local_path)
onnx.checker.check_model(onnx_model)
print('ONNX model check passed.')

/tmp/ipykernel_5420/1270963204.py:11: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0407 01:43:49.066000 5420 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `Deepfake_1DCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `Deepfake_1DCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 120, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 115, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 4 of general pattern rewrite rules.
ONNX model exported to Drive: /content/drive/MyDrive/deepfake_1dcnn_web.onnx
ONNX model check passed.


In [14]:
# ── Cross-dataset evaluation: ASVspoof model vs ITW eval ──────────────────────
import optuna

# Get best params from db
study = optuna.load_study(
    study_name='1dcnn_asvspoof',
    storage='sqlite:////content/drive/MyDrive/optuna_1dcnn.db'
)
best = study.best_params
print('Best params:', best)

# Reload ASVspoof trained model with correct params
norm_stats = torch.load('/content/drive/MyDrive/1dcnn_norm_stats.pth')
asv_model = Deepfake_1DCNN(
    norm_stats['means'], norm_stats['stds'],
    out_channels=best['out_channels'],
    kernel_size=best['kernel_size'],
    dilation=best['dilation'],
    dropout=best['dropout'],
    fc1_out=best['fc1_out'],
).to(device)
asv_model.load_state_dict(torch.load('/content/drive/MyDrive/1dcnn_best.pth'))
asv_model.eval()
print('ASVspoof model reloaded.')

# Load ITW eval features
itw_eval_df = pd.read_pickle('/content/drive/MyDrive/ITW_Features_eval.pkl')
X_itw_eval = np.stack(itw_eval_df['1dcnn_features'].values).transpose(0, 2, 1).astype(np.float32)
y_itw_eval = itw_eval_df['label'].values

itw_eval_ds = TensorDataset(torch.from_numpy(X_itw_eval), torch.from_numpy(y_itw_eval))
itw_eval_dl = DataLoader(itw_eval_ds, batch_size=128, shuffle=False,
                         pin_memory=True, num_workers=0)

# Evaluate
cross_eer, cross_fig = eval_model(
    asv_model, itw_eval_dl, device,
    'Cross-Dataset: ASVspoof Model vs ITW Eval',
    show_plots=True
)

print(f'\nASVspoof model on ITW eval EER: {cross_eer:.4f}%')

Best params: {'batch_size': 128, 'lr': 0.00733005223070758, 'weight_decay': 1.3507337007151238e-05, 'out_channels': 64, 'kernel_size': 7, 'dilation': 2, 'dropout': 0.26981219618819724, 'fc1_out': 32, 'optimizer': 'Ranger', 'max_lr_factor': 2.7134491709710105}
ASVspoof model reloaded.

========== Cross-Dataset: ASVspoof Model vs ITW Eval Results ==========
EER:       84.8101%
Accuracy:  0.5114
F1 Score:  0.6768
Precision: 0.5114
Recall:    1.0000

Classification Report:
              precision    recall  f1-score   support

    Bonafide       0.00      0.00      0.00      2264
       Spoof       0.51      1.00      0.68      2370

    accuracy                           0.51      4634
   macro avg       0.26      0.50      0.34      4634
weighted avg       0.26      0.51      0.35      4634


ASVspoof model on ITW eval EER: 84.8101%


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
